In [ ]:
from pathlib import Path
import pandas as pd

project_root = next(
    folder for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "src" / "ventas_app").is_dir()
)
DATA_PATH = project_root / "data" / "ventas.csv"
DATA_DIR = project_root / "data" / "processed" / "ventas"
DATA_DIR.mkdir(parents=True, exist_ok=True)
ventas = pd.read_csv(DATA_PATH)
ventas.head()


In [ ]:
print("Número de filas:", ventas.shape)
print("Tipos de datos de las columnas:", ventas.dtypes)
print("Número de valores nulos por columna:", ventas.isnull().sum())

Las filas invalidas tienen unidades o precio ausentes, no numericos o no positivos.
La validacion las separa antes de calcular importes; el checkpoint es 140 validas y 10 invalidas.

In [ ]:
ventas.head()

In [ ]:
def cargar_ventas(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"No existe el fichero: {path}")
    return pd.read_csv(path)


raw = cargar_ventas(DATA_PATH)
raw

In [ ]:
def validar_ventas(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    work = frame.copy()

    work["unidades"] = pd.to_numeric(work["unidades"], errors="coerce")
    work["precio_unitario"] = pd.to_numeric(work["precio_unitario"], errors="coerce")

    ok = (
        work["unidades"].notna()
        & (work["unidades"] > 0)
        & work["precio_unitario"].notna()
        & (work["precio_unitario"] > 0)
    )

    validos = work.loc[ok].copy()
    errores = work.loc[~ok].copy()
    validos["importe"] = validos["unidades"] * validos["precio_unitario"]
    return validos, errores


validos, errores = validar_ventas(raw)
print("Filas válidas:", len(validos))
print("Filas inválidas:", len(errores))

In [ ]:
importe_por_region = (
    validos.groupby("region", as_index=False)["importe"].sum().sort_values("importe", ascending=False)
)
importe_por_region


In [ ]:
top_productos = (
    validos.groupby("producto", as_index=False)["importe"].sum().sort_values("importe", ascending=False).head(3)
)
top_productos

In [ ]:
compras_por_cliente = validos["cliente_id"].value_counts()
clientes_recurrentes = compras_por_cliente[compras_por_cliente > 1]
clientes_recurrentes

In [ ]:
import json

salida_csv = DATA_DIR / "ventas_limpias.csv"
salida_json = DATA_DIR / "calidad_datos.json"

validos.to_csv(salida_csv, index=False)

calidad = {
    "filas_totales": int(len(raw)),
    "filas_validas": int(len(validos)),
    "filas_invalidas": int(len(errores)),
    "importe_total": float(validos["importe"].sum()),
}
salida_json.write_text(json.dumps(calidad, indent=2, ensure_ascii=False), encoding="utf-8")

print("escrito:", salida_csv)
print("escrito:", salida_json)

calidad